# Study 885 — Ultra-Short Credit Pickup — the teardown

The excess-of-bills Sharpe race, the HAC *t* on the credit-minus-bill pickup, a block-bootstrap Sharpe CI, the sub-era cut, the drawdown & stress windows, MINT's long 2009→ history, the cost math, and the synthetic control.

In [1]:
R = {'fingerprint': '22e1cddb739d', 'asof': '2026-06-30', 'n_days': 2289, 'start': '2017-05-22', 'end': '2026-06-30', 'years': 9.08, 'jpst_sharpe': 0.61, 'icsh_sharpe': 0.54, 'mint_sharpe': 0.4, 'shv_sharpe': 0.11, 'jpst_ret': 3.0, 'icsh_ret': 2.94, 'mint_ret': 2.81, 'bil_ret': 2.41, 'shv_ret': 2.44, 'jpst_vol': 0.93, 'icsh_vol': 0.97, 'mint_vol': 0.98, 'bil_vol': 0.25, 'shv_vol': 0.27, 'sleeve_bps': 49.8, 'sleeve_t': 1.3, 'sleeve_sharpe': 0.62, 'sleeve_lags': 8, 'jpst_bps': 57.9, 'jpst_pt': 1.6, 'icsh_bps': 52.0, 'icsh_pt': 1.46, 'mint_bps': 39.5, 'mint_pt': 0.84, 'shv_bps': 2.8, 'ci_lo': -0.26, 'ci_hi': 2.21, 'ci_fracneg': 0.091, 'ci_block': 13, 'early_bps': 49.2, 'early_t': 3.21, 'early_n': 406, 'late_bps': 49.9, 'late_t': 1.08, 'late_n': 1883, 'welch_t': -0.02, 'bil_dd': -0.21, 'jpst_dd': -3.28, 'icsh_dd': -3.94, 'mint_dd': -4.62, 'y2022_bil': 1.4, 'y2022_jpst': 1.14, 'y2022_icsh': 0.96, 'y2022_mint': -1.01, 'covid_bil': 0.26, 'covid_jpst': -2.75, 'covid_icsh': -3.55, 'covid_mint': -4.34, 'mint_long_bps': 77.0, 'mint_long_t': 2.82, 'mint_long_sharpe': 0.89, 'mint_long_cilo': 0.18, 'mint_long_cihi': 1.92, 'mint_long_n': 4177, 'mint_long_years': 16.6, 'mint_long_early_t': 7.44, 'mint_long_late_t': 0.73, 'cost1_net': 47.8, 'cost1_sharpe': 0.59, 'cost2_net': 45.8, 'cost5_net': 39.8, 'syn_null_tmean': 0.61, 'syn_null_fire': '0/12', 'syn_plant': 120.0, 'syn_plant_tmean': 3.51, 'syn_plant_fire': '11/12', 'syn_plant_recovered': 120.0}

## 1. The reward-per-risk race — annualised EXCESS-of-BIL Sharpe (rf = BIL)

Every Sharpe is excess-of-cash (minus BIL). The credit sleeve genuinely wins on the point estimate; SHV (a hair more duration, ~zero credit) barely moves.

In [2]:
print(f"JPST : excess Sharpe {R['jpst_sharpe']:+.2f}  (ret {R['jpst_ret']:.2f}%/yr, vol {R['jpst_vol']:.2f}%)")
print(f"ICSH : excess Sharpe {R['icsh_sharpe']:+.2f}  (ret {R['icsh_ret']:.2f}%/yr, vol {R['icsh_vol']:.2f}%)")
print(f"MINT : excess Sharpe {R['mint_sharpe']:+.2f}  (ret {R['mint_ret']:.2f}%/yr, vol {R['mint_vol']:.2f}%)")
print(f"SHV  : excess Sharpe {R['shv_sharpe']:+.2f}  (ret {R['shv_ret']:.2f}%/yr, vol {R['shv_vol']:.2f}%)")
print(f"BIL  : excess Sharpe  0.00  (ret {R['bil_ret']:.2f}%/yr, vol {R['bil_vol']:.2f}%)  <- the cash leg")

JPST : excess Sharpe +0.61  (ret 3.00%/yr, vol 0.93%)
ICSH : excess Sharpe +0.54  (ret 2.94%/yr, vol 0.97%)
MINT : excess Sharpe +0.40  (ret 2.81%/yr, vol 0.98%)
SHV  : excess Sharpe +0.11  (ret 2.44%/yr, vol 0.27%)
BIL  : excess Sharpe  0.00  (ret 2.41%/yr, vol 0.25%)  <- the cash leg


## 2. The pickup — equal-weight credit sleeve minus BIL, HAC t

Ultra-short credit total returns are serially correlated (smooth NAV marks), so the Newey-West correction knocks the naive t down — this is why a 0.62 Sharpe over 9 years does *not* translate into a t≥2.

In [3]:
print(f"sleeve - BIL : {R['sleeve_bps']:+.1f} bps/yr  HAC t = {R['sleeve_t']:+.2f}  "
      f"(n={R['n_days']}, lags={R['sleeve_lags']}, excess Sharpe {R['sleeve_sharpe']:+.2f})")
print(f"  JPST-BIL {R['jpst_bps']:+.1f} (t={R['jpst_pt']:+.2f})  "
      f"ICSH-BIL {R['icsh_bps']:+.1f} (t={R['icsh_pt']:+.2f})  "
      f"MINT-BIL {R['mint_bps']:+.1f} (t={R['mint_pt']:+.2f})")
print(f"  SHV-BIL {R['shv_bps']:+.1f} bps/yr (the near-zero-credit control)")

sleeve - BIL : +49.8 bps/yr  HAC t = +1.30  (n=2289, lags=8, excess Sharpe +0.62)
  JPST-BIL +57.9 (t=+1.60)  ICSH-BIL +52.0 (t=+1.46)  MINT-BIL +39.5 (t=+0.84)
  SHV-BIL +2.8 bps/yr (the near-zero-credit control)


## 3. Bootstrap CI on the sleeve excess Sharpe (circular block, 2000 draws)

The interval crosses zero — the pickup is not distinguishable from cash at 95%.

In [4]:
print(f"sharpe {R['sleeve_sharpe']:+.2f}  95% CI [{R['ci_lo']:+.2f}, {R['ci_hi']:+.2f}]  "
      f"frac<0 = {R['ci_fracneg']:.3f}  (block {R['ci_block']})")

sharpe +0.62  95% CI [-0.26, +2.21]  frac<0 = 0.091  (block 13)


## 4. Sub-eras — the edge lives in the past

Split the sleeve pickup at 2019, and split MINT's long history at 2018. The *mean* is stable, but the significance evaporates in the modern regime (post-GFC spreads compressed; 2021 ZIRP left no spread to earn; 2022 duration hurt).

In [5]:
print(f"sleeve <2019 : {R['early_bps']:+.1f} bps/yr  HAC t = {R['early_t']:+.2f}  (n={R['early_n']})")
print(f"sleeve >=2019: {R['late_bps']:+.1f} bps/yr  HAC t = {R['late_t']:+.2f}  (n={R['late_n']})  Welch t={R['welch_t']:+.2f}")
print(f"MINT long-history {R['mint_long_years']:.0f}y : {R['mint_long_bps']:+.0f} bps/yr  HAC t = {R['mint_long_t']:+.2f}  "
      f"Sharpe {R['mint_long_sharpe']:.2f}  CI [{R['mint_long_cilo']:+.2f},{R['mint_long_cihi']:+.2f}]")
print(f"  MINT <2018 t = {R['mint_long_early_t']:+.2f}   vs   MINT >=2018 t = {R['mint_long_late_t']:+.2f}")

sleeve <2019 : +49.2 bps/yr  HAC t = +3.21  (n=406)
sleeve >=2019: +49.9 bps/yr  HAC t = +1.08  (n=1883)  Welch t=-0.02
MINT long-history 17y : +77 bps/yr  HAC t = +2.82  Sharpe 0.89  CI [+0.18,+1.92]
  MINT <2018 t = +7.44   vs   MINT >=2018 t = +0.73


## 5. It is not riskless — drawdowns & stress windows

In [6]:
print(f"max DD (common sample): BIL {R['bil_dd']:.2f}%  JPST {R['jpst_dd']:.2f}%  "
      f"ICSH {R['icsh_dd']:.2f}%  MINT {R['mint_dd']:.2f}%")
print(f"COVID Feb-Mar 2020    : BIL {R['covid_bil']:+.2f}%  JPST {R['covid_jpst']:+.2f}%  "
      f"ICSH {R['covid_icsh']:+.2f}%  MINT {R['covid_mint']:+.2f}%")
print(f"2022 hiking year      : BIL {R['y2022_bil']:+.2f}%  JPST {R['y2022_jpst']:+.2f}%  "
      f"ICSH {R['y2022_icsh']:+.2f}%  MINT {R['y2022_mint']:+.2f}%")

max DD (common sample): BIL -0.21%  JPST -3.28%  ICSH -3.94%  MINT -4.62%
COVID Feb-Mar 2020    : BIL +0.26%  JPST -2.75%  ICSH -3.55%  MINT -4.34%
2022 hiking year      : BIL +1.40%  JPST +1.14%  ICSH +0.96%  MINT -1.01%


## 6. Costs — the trade is buy-and-hold, so friction is trivial

You buy the sleeve once; maturities roll inside the fund; ETF fees are already in the net-of-fee tape. One-way spread × NAV × ~1 turnover/yr barely registers — which is exactly why the *Signal*, not costs, is the binding constraint here.

In [7]:
print(f"1 bp one-way x1/yr: net {R['cost1_net']:+.1f} bps/yr (Sharpe {R['cost1_sharpe']:+.2f})")
print(f"2 bp one-way x1/yr: net {R['cost2_net']:+.1f} bps/yr")
print(f"5 bp one-way x1/yr: net {R['cost5_net']:+.1f} bps/yr  (gross was {R['sleeve_bps']:+.1f})")

1 bp one-way x1/yr: net +47.8 bps/yr (Sharpe +0.59)
2 bp one-way x1/yr: net +45.8 bps/yr
5 bp one-way x1/yr: net +39.8 bps/yr  (gross was +49.8)


## 7. Synthetic control — the machinery is unbiased (never market evidence)

Live: the detector must NOT fire on the null and must recover a planted pickup by the exact planted amount.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from ultra_short import data, strategy as st
def stats(planted, seed):
    w = data.synthetic_world(pickup_bps_yr=planted, seed=seed, n_days=2000)
    ex = (w['CREDIT'] - w['CASH']).dropna()
    return st.hac_mean(ex)['t_nw'], st.hac_mean(ex)['mean_bps_yr']
null_t = np.array([stats(0.0, 885+s)[0] for s in range(12)])
plant = [stats(120.0, 885+s) for s in range(12)]
plant_t = np.array([p[0] for p in plant])
m0 = np.mean([stats(0.0, 885+s)[1] for s in range(12)])
m1 = np.mean([p[1] for p in plant])
print('null   (0 bps/yr): mean t %+.2f, |t|>=2 in %d/12' % (null_t.mean(), int((abs(null_t)>=2).sum())))
print('planted (+120)   : mean t %+.2f, |t|>=2 in %d/12' % (plant_t.mean(), int((abs(plant_t)>=2).sum())))
print('recovered carry  : %+.1f bps/yr (planted +120.0)' % (m1 - m0))

null   (0 bps/yr): mean t +0.61, |t|>=2 in 0/12
planted (+120)   : mean t +3.51, |t|>=2 in 11/12
recovered carry  : +120.0 bps/yr (planted +120.0)


## Verdict

- **Signal — Weak.** The ultra-short credit pickup is genuinely there in the point estimate — the sleeve out-earns bills by **+49.8 bps/yr** at an excess Sharpe of **0.62** (vs 0.11 for SHV, 0 for BIL), and on MINT's 17-year tape it is **+77 bps/yr at HAC *t* = 2.82** with a bootstrap CI clear of zero. But it **fails the robustness bar**: the full-sleeve HAC *t* is only **1.30**, the bootstrap Sharpe CI crosses zero ([-0.26, +2.21], 9% negative), and it does **not hold across sub-eras** — the significance is entirely a pre-2018 post-GFC phenomenon (MINT *t* = 7.44 → 0.73). Sign is right and economically real everywhere, so not None; robustness fails, so not Real. *Young ETFs → short live history.*
- **Tradability — Fragile.** Costs are trivial (buy-and-hold, fees inside the tape; at 1 bp the net is +47.8 of the +49.8 gross) — so this is **not** a cost Mirage. It is Fragile because the edge is thin (~50 bps/yr), era-contingent (dead in the modern regime), and **not riskless**: MINT lost -1.0% in 2022 while bills made +1.4%, and the sleeve drew down -4.6% vs bills' -0.21% in the COVID crunch. A real-but-thin carry you cannot certify → Fragile.